# 🏅 Golden Beta Builder

Constitue le goldset `golden_beta` à partir des questions du beta test (beta_v2, beta_v3).

## Pipeline
1. Charger les questions beta_v2 + beta_v3 depuis `goldset_questions_v2`
2. Joindre avec `chat_feedbacks` pour récupérer `error_category` et `beta_scope`
3. Filtrer : exclure "Document manquant" et "Hors périmètre" (beta_scope = Non)
4. Récupérer les réponses beta réelles depuis `goldset_runs` (v2_prod, v3_prod)
5. Récupérer les feedbacks utilisateurs (commentaires, notes)
6. Tagger `golden_beta` dans goldset_questions_v2

## Objectif
Avoir un subset propre de questions beta pour :
- Relancer retrieval + génération avec la config optimale V3
- Comparer avec les réponses beta réelles via LLM Judge
- Mesurer le progrès objectif entre le beta test et la config actuelle

In [ ]:
import os, sys, json
from pathlib import Path
from collections import Counter

import pandas as pd
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

DSN = os.getenv("TUNNEL_DSN") or os.getenv("SCALINGO_POSTGRESQL_URL") or os.getenv("PG_DSN")
conn = psycopg.connect(DSN, row_factory=dict_row)
cur = conn.cursor()
print("Connected ✅")

## 1. Charger les questions beta et les feedbacks associés

In [ ]:
# Load beta questions with feedback data
cur.execute("""
    SELECT
        gq.id,
        gq.question,
        gq.gold_answer,
        gq.gold_sources,
        gq.theme,
        gq.tags,
        gq.difficulty,
        gq.goldset_name,
        gq.original_turn_id,
        -- Feedback info (join via original_turn_id -> chat_runs -> chat_feedbacks)
        cf.error_category,
        cf.beta_scope,
        cf.helpful,
        cf.stars,
        cf.comment,
        cf.reasons_positive,
        cf.reasons_negative,
        cf.ai_reason,
        -- Chat run info
        cr.answer AS beta_answer,
        cr.v3_reformulated_query,
        cr.v3_detected_theme,
        cr.v3_intent,
        cr.rag_version AS beta_rag_version
    FROM goldset_questions_v2 gq
    LEFT JOIN chat_runs cr ON cr.turn_id::text LIKE gq.original_turn_id || '%%'
        AND gq.original_turn_id IS NOT NULL AND gq.original_turn_id != ''
    LEFT JOIN chat_feedbacks cf ON cf.turn_id = cr.turn_id
    WHERE gq.goldset_name IN ('beta_v2', 'beta_v3')
    ORDER BY gq.goldset_name, gq.id
""")
all_beta = cur.fetchall()
df = pd.DataFrame(all_beta)

print(f"Total beta questions: {len(df)}")
print(f"\nBy goldset_name:")
print(df['goldset_name'].value_counts().to_string())
print(f"\nWith feedback: {df['error_category'].notna().sum()}")
print(f"With beta answer: {df['beta_answer'].notna().sum()}")
print(f"With gold_answer: {df['gold_answer'].notna().sum()}")
print(f"With gold_sources: {(df['gold_sources'].notna() & (df['gold_sources'] != '')).sum()}")
print(f"With reformulated query: {df['v3_reformulated_query'].notna().sum()}")

In [ ]:
# Analyze exclusion criteria
print("=" * 60)
print("EXCLUSION ANALYSIS")
print("=" * 60)

# 1. Document manquant
mask_missing = df['error_category'] == 'missing_document'
n_missing = mask_missing.sum()
print(f"\n📄 Document manquant: {n_missing}")
if n_missing > 0:
    for _, r in df[mask_missing].head(5).iterrows():
        print(f"  Q{r['id']}: {r['question'][:70]}...")

# 2. Hors périmètre (beta_scope = Non)
mask_hors = df['beta_scope'].fillna('').astype(str).str.strip().str.lower() == 'non'
n_hors = mask_hors.sum()
print(f"\n🚫 Hors périmètre (beta_scope=Non): {n_hors}")
if n_hors > 0:
    for _, r in df[mask_hors].head(5).iterrows():
        print(f"  Q{r['id']}: {r['question'][:70]}...")

# 3. Questions without any feedback (might still be valid)
mask_no_feedback = df['error_category'].isna() & df['beta_scope'].isna()
n_no_feedback = mask_no_feedback.sum()
print(f"\n⬜ Sans feedback: {n_no_feedback}")

# 3. Questions with NaN text (invalid)
mask_nan_question = df['question'].isna() | (df['question'].astype(str).str.strip() == '')
n_nan = mask_nan_question.sum()
print(f"\n🚫 Question NaN/vide: {n_nan}")
if n_nan > 0:
    for _, r in df[mask_nan_question].head(5).iterrows():
        print(f"  Q{r['id']}: question='{r['question']}'")

# Combined exclusion
exclude_mask = mask_missing | mask_hors | mask_nan_question
n_excluded = exclude_mask.sum()
n_kept = len(df) - n_excluded
print(f"\n{'=' * 60}")
print(f"Total à exclure: {n_excluded}")
print(f"Total à garder (golden_beta): {n_kept}")

In [ ]:
# Apply filters
df_golden = df[~exclude_mask].copy()

print(f"🏅 Golden Beta: {len(df_golden)} questions")
print(f"\nBy goldset_name:")
print(df_golden['goldset_name'].value_counts().to_string())
print(f"\nError categories (for kept questions):")
ec = df_golden['error_category'].value_counts(dropna=False)
for cat, cnt in ec.items():
    print(f"  {cat if pd.notna(cat) else 'No feedback'}: {cnt}")
print(f"\nThemes:")
themes = df_golden['theme'].value_counts(dropna=False)
for t, cnt in themes.head(15).items():
    print(f"  {t if pd.notna(t) else 'None'}: {cnt}")
print(f"\nIntents from beta:")
intents = df_golden['v3_intent'].value_counts(dropna=False)
for i, cnt in intents.items():
    print(f"  {i if pd.notna(i) else 'None'}: {cnt}")

## 2. Vérifier les questions à reformuler

Certaines questions beta sont des follow-ups qui ont été reformulées par l'intent gater.
On vérifie si la question dans le goldset est déjà la reformulée ou l'originale.

In [ ]:
# Check reformulations
# Only keep rows where v3_reformulated_query is non-null AND non-empty/whitespace
reformulated = df_golden[
    df_golden['v3_reformulated_query'].notna() &
    (df_golden['v3_reformulated_query'].astype(str).str.strip() != '')
].copy()
print(f"Questions avec reformulation non-vide dans les logs: {len(reformulated)}")

if len(reformulated) > 0:
    # Among those with actual reformulation, check which differ from the original
    n_different = 0
    print("\nExamples de reformulations différentes:")
    for _, r in reformulated.iterrows():
        q_orig = str(r['question']).strip()
        q_reform = str(r['v3_reformulated_query']).strip()
        if q_orig != q_reform:
            n_different += 1
            if n_different <= 5:
                print(f"\n  Q{r['id']}:")
                print(f"    Goldset:      {q_orig[:80]}")
                print(f"    Reformulated: {q_reform[:80]}")
    
    n_same = len(reformulated) - n_different
    print(f"\n📊 Bilan:")
    print(f"  Reformulation identique à la question: {n_same}")
    print(f"  Reformulation différente: {n_different}")
    if n_different > 0:
        print(f"  → Pour ces {n_different} questions, on utilisera la reformulation comme query de retrieval")
else:
    print("Aucune reformulation trouvée dans les logs")

## 3. Récupérer les réponses beta existantes depuis goldset_runs

On a déjà les runs beta dans `goldset_runs` (configs v2_prod, v3_prod).
On les récupère pour le Judge 2 (comparaison beta vs optimal).

In [ ]:
# Check existing beta runs in goldset_runs
golden_ids = df_golden['id'].tolist()
if golden_ids:
    placeholders = ','.join(['%s'] * len(golden_ids))
    cur.execute(f"""
        SELECT config_name, COUNT(*) as cnt,
               COUNT(*) FILTER (WHERE response IS NOT NULL) as has_response,
               COUNT(*) FILTER (WHERE retrieved_context IS NOT NULL) as has_context
        FROM goldset_runs
        WHERE question_id IN ({placeholders})
        GROUP BY config_name
        ORDER BY config_name
    """, golden_ids)
    existing = cur.fetchall()
    print("Existing runs for golden_beta questions:")
    for r in existing:
        print(f"  {r['config_name']:25s} {r['cnt']:4d} runs | {r['has_response']:4d} responses | {r['has_context']:4d} with context")
else:
    print("No golden_beta questions to check")

In [ ]:
# Load beta responses for comparison (for Judge 2)
# We want the original beta answer + user feedback for each golden_beta question
beta_responses = {}

if golden_ids:
    placeholders = ','.join(['%s'] * len(golden_ids))
    cur.execute(f"""
        SELECT question_id, config_name, response, retrieved_context,
               retrieval_time_ms, generation_time_ms
        FROM goldset_runs
        WHERE question_id IN ({placeholders})
          AND config_name IN ('v2_prod', 'v3_prod')
        ORDER BY question_id, config_name
    """, golden_ids)
    for row in cur.fetchall():
        qid = row['question_id']
        if qid not in beta_responses:
            beta_responses[qid] = {}
        beta_responses[qid][row['config_name']] = {
            'response': row['response'],
            'has_context': row['retrieved_context'] is not None,
        }

n_with_beta = sum(1 for qid in golden_ids if qid in beta_responses)
print(f"\nGolden beta questions with existing beta responses: {n_with_beta}/{len(golden_ids)}")
print(f"  v2_prod: {sum(1 for v in beta_responses.values() if 'v2_prod' in v)}")
print(f"  v3_prod: {sum(1 for v in beta_responses.values() if 'v3_prod' in v)}")

## 4. Préparer le dataset golden_beta

On crée un DataFrame final avec toutes les infos nécessaires pour :
- Le retrieval + génération optimale
- Le Judge 1 (error categorization)
- Le Judge 2 (beta comparison)

In [ ]:
# Build the final golden_beta dataset
# Safety: exclude NaN/empty questions + questions without feedback
n_before = len(df_golden)
df_golden = df_golden[
    df_golden['question'].notna() &
    (df_golden['question'].astype(str).str.strip() != '') &
    (df_golden['question'].astype(str).str.lower() != 'nan')
].copy()
n_nan_removed = n_before - len(df_golden)

n_before2 = len(df_golden)
df_golden = df_golden[df_golden['stars'].notna()].copy()
n_no_feedback = n_before2 - len(df_golden)

print(f"Cleanup: {n_nan_removed} questions NaN retirées, {n_no_feedback} sans feedback retirées")
print(f"→ {len(df_golden)} questions retenues")

records = []
for _, r in df_golden.iterrows():
    qid = r['id']
    
    # Determine retrieval query: use reformulation if available and different
    retrieval_query = r['question']
    was_reformulated = False
    if pd.notna(r.get('v3_reformulated_query')):
        reform = r['v3_reformulated_query'].strip()
        if reform and reform != r['question'].strip():
            retrieval_query = reform
            was_reformulated = True
    
    # Get beta response (prefer v3_prod, fallback v2_prod)
    beta_resp = beta_responses.get(qid, {})
    beta_config = 'v3_prod' if 'v3_prod' in beta_resp else ('v2_prod' if 'v2_prod' in beta_resp else None)
    beta_response_text = beta_resp.get(beta_config, {}).get('response') if beta_config else r.get('beta_answer')
    
    records.append({
        'question_id': qid,
        'question': r['question'],
        'retrieval_query': retrieval_query,
        'was_reformulated': was_reformulated,
        'goldset_name': r['goldset_name'],
        'theme': r.get('theme'),
        'tags': r.get('tags'),
        'gold_answer': r.get('gold_answer'),
        'gold_sources': r.get('gold_sources'),
        # Beta test info
        'beta_answer': beta_response_text,
        'beta_config': beta_config,
        'beta_rag_version': r.get('beta_rag_version'),
        # User feedback
        'feedback_stars': r.get('stars'),
        'feedback_helpful': r.get('helpful'),
        'feedback_comment': r.get('comment'),
        'feedback_reasons_positive': r.get('reasons_positive'),
        'feedback_reasons_negative': r.get('reasons_negative'),
        'feedback_error_category': r.get('error_category'),
        'feedback_ai_reason': r.get('ai_reason'),
    })

df_final = pd.DataFrame(records)
print(f"🏅 Golden Beta dataset: {len(df_final)} questions")
print(f"  With beta answer: {df_final['beta_answer'].notna().sum()}")
print(f"  With gold_answer: {df_final['gold_answer'].notna().sum()}")
print(f"  With feedback: {df_final['feedback_stars'].notna().sum()}")
print(f"  Reformulated for retrieval: {df_final['was_reformulated'].sum()}")
print(f"  With gold_sources: {(df_final['gold_sources'].notna() & (df_final['gold_sources'] != '')).sum()}")

In [ ]:
# Preview
display_cols = ['question_id', 'question', 'goldset_name', 'theme',
                'was_reformulated', 'feedback_stars', 'feedback_error_category',
                'beta_config']
df_final[display_cols].head(20)

---

# Phase 2 : Génération avec la config optimale V3

Le goldset `golden_beta` est prêt. On utilise maintenant le notebook existant
`goldset_multiconfig_generation2_v3run.ipynb` pour lancer le retrieval + génération.

## Configuration à appliquer

Ouvrir le notebook `goldset_multiconfig_generation2_v3run.ipynb` et configurer :

```python
CONFIG_MODE = "custom"
CONFIG_NAME = "v3_optimal_golden_beta"

GOLDSET_FILTER = {
    "goldset_name": None,
    "theme": None,
    "limit": None,
    "has_gold_answer": False,
    "tags": ["golden_beta"],  # ← AJOUTER support tags
}

CUSTOM_CONFIG = {
    "context_mode": "standard",
    "search_mode": "semantic",
    "embedding_model": "albert",
    "initial_top_k": 50,             # ← 50 pour maximiser le recall
    "token_budget": 8000,
    "enable_escalation": False,      # ← désactivé pour ce test
    "enable_selector": True,
    "selector_model": "openweight-medium",
    "generator_model": "openweight-large",
    "temperature": 0.0,
    "enable_section_reranker": True,
    "section_rerank_top_k": 20,      # ← 20 sections après rerank
    "system_prompt_name": "system_prompt_V6_optimized.md",
    "enable_intent_gating": False,   # ← désactivé (goldset curé)
    "enable_acronym_expansion": True,
    # DE tables à brancher
    "de_tables": ["rag_chunks_rgrh", "rag_chunks_dgafp"],
}
```

## Checklist
- [ ] Ajouter `tags` à `load_goldset_questions()` dans le notebook de génération
- [ ] Ajouter `de_tables` dans `build_custom_config()`
- [ ] Lancer la génération
- [ ] Calculer les métriques RAGAS (faithfulness uniquement, pas de gold_answer)
- [ ] Lancer les 2 LLM Judges (notebook séparé)

## 5. Tagger golden_beta dans goldset_questions_v2

In [ ]:
GOLDEN_BETA_TAG = "golden_beta"

golden_ids = df_final['question_id'].tolist()
print(f"Questions to tag: {len(golden_ids)}")

# Remove old tags
cur.execute("""
    UPDATE goldset_questions_v2
    SET tags = array_remove(tags, %s)
    WHERE tags @> ARRAY[%s]
""", [GOLDEN_BETA_TAG, GOLDEN_BETA_TAG])
removed = cur.rowcount
print(f"Removed old '{GOLDEN_BETA_TAG}' tag from {removed} questions")

# Add tag
if golden_ids:
    placeholders = ','.join(['%s'] * len(golden_ids))
    cur.execute(f"""
        UPDATE goldset_questions_v2
        SET tags = CASE
            WHEN tags IS NULL THEN ARRAY[%s]
            ELSE array_append(tags, %s)
        END
        WHERE id IN ({placeholders})
    """, [GOLDEN_BETA_TAG, GOLDEN_BETA_TAG] + golden_ids)
    tagged = cur.rowcount
    print(f"Tagged {tagged} questions with '{GOLDEN_BETA_TAG}'")

conn.commit()
print("\n✅ golden_beta tag saved!")

In [ ]:
# Save the enriched dataset as CSV for reference
export_path = "golden_beta_dataset.csv"
df_final.to_csv(export_path, index=False)
print(f"Exported to {export_path}")
print(f"Columns: {list(df_final.columns)}")

## 6. Vérification et stats finales

In [ ]:
# Verify
cur.execute("""
    SELECT COUNT(*) as cnt FROM goldset_questions_v2
    WHERE tags @> ARRAY['golden_beta']
""")
count = cur.fetchone()['cnt']
print(f"🏅 Golden Beta: {count} questions tagged in goldset_questions_v2")

# Feedback distribution
print("\n📊 Feedback distribution:")
print(f"  Stars distribution:")
for stars, cnt in df_final['feedback_stars'].value_counts(dropna=False).sort_index().items():
    label = f"{stars}⭐" if pd.notna(stars) else "No feedback"
    print(f"    {label}: {cnt}")

print(f"\n  Error categories (for questions with feedback):")
for cat, cnt in df_final['feedback_error_category'].value_counts(dropna=False).items():
    label = cat if pd.notna(cat) else 'No error / No feedback'
    print(f"    {label}: {cnt}")

print(f"\n💡 Prochaines étapes:")
print(f"   1. Lancer le notebook de génération avec CONFIG_NAME='v3_optimal'")
print(f"      GOLDSET_FILTER = {{'tags': ['golden_beta']}}")
print(f"   2. Calculer les métriques RAGAS (faithfulness)")
print(f"   3. Lancer Judge 1 (Error Categorization) sur les runs mal notés")
print(f"   4. Lancer Judge 2 (Beta Comparison) sur tous les runs")

## 7. Résumé

Le golden_beta est constitué. Pour la suite :

### Notebook de génération (étape 2)
Dupliquer `goldset_multiconfig_generation2_v3run.ipynb` avec :
- `CONFIG_MODE = "custom"`
- `CONFIG_NAME = "v3_optimal"`
- Config: top_k=50, section_rerank=20, selector=medium, +RGRH+DGAFP, generator=openweight-large
- Filtrer par tag `golden_beta`

### Judge 1 — Error Categorization (étape 3)
Pour chaque run avec mauvaise note RAGAS :
- Input: query, context PRÉ-selector, selector reasoning, context POST-selector, response
- Output: catégorie d'erreur (retrieval_miss, selector_error, generator_hallucination, etc.)

### Judge 2 — Beta Comparison (étape 4)
Pour chaque question :
- Input: query, beta_answer, user_feedback, new_optimal_answer
- Output: problème résolu?, qualité (meilleure/égale/inférieure)